# Problem 7 — Pencocokan kurva terbuka dengan Python

**ID masalah:** `O005-LEGA-V101-CH01-P07`  
**Benih acak tetap:** `20260821`

Teks masalah sumber meminta pembaca membuat titik data yang dekat dengan grafik empat fungsi berikut, mencoba pencocokan kurva, lalu menilai apakah hasilnya memuaskan:

1. $f(x)=3x+5$
2. $f(x)=7x$
3. $f(x)=e^{3x+7}$
4. $f(x)=e^{7x}$

Notebook ini adalah pengganti terbuka yang ditulis secara independen untuk adaptasi Bahasa Indonesia. Implementasi memakai NumPy, SciPy, dan Matplotlib; tidak ada kode, data, atau antarmuka perangkat lunak proprieter yang disalin. Teks masalah sumber berasal dari Joceline Lega, *Introduction to Mathematical Modeling*, versi 1.01, CC BY-NC-SA 4.0. Notebook baru ini didistribusikan bersama adaptasi dengan lisensi yang sama; perubahan meliputi data sintetis, implementasi Python, visualisasi, dan pemeriksaan otomatis. Tidak ada dukungan atau pengesahan oleh penulis maupun University of Arizona yang tersirat.

## Metode

Untuk setiap fungsi, kita membangkitkan 28 titik sintetis dengan gangguan kecil dan benih acak tetap. Kita lalu mencocokkan dua keluarga model yang sama-sama mempunyai dua parameter:

- model linear $y=ax+b$;
- model eksponensial $y=e^{ax+b}$.

Perbandingan memakai galat akar rata-rata kuadrat (RMSE), $R^2$, kurva hasil pencocokan, dan pola residual $r_i=y_i-\hat y_i$. Karena kedua calon mempunyai jumlah parameter yang sama, RMSE dapat dibandingkan langsung. Residual yang kecil dan tersebar tanpa pola sistematis merupakan tanda kecocokan yang lebih baik.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import curve_fit

SEED = 20260821
np.set_printoptions(precision=5, suppress=True)

print(f"Benih acak: {SEED}")
print(f"NumPy {np.__version__}")

In [ ]:
def model_linear(x, a, b):
    return a * x + b


def model_eksponensial(x, a, b):
    return np.exp(a * x + b)


def buat_data(seed=SEED):
    """Bangkitkan empat himpunan data sintetis secara deterministik."""
    rng = np.random.default_rng(seed)
    spesifikasi = [
        {
            "id": "O005-LEGA-V101-CH01-P07-D1",
            "label": "f(x) = 3x + 5",
            "x": np.linspace(-1.25, 2.50, 28),
            "fungsi": lambda x: 3.0 * x + 5.0,
            "bising": "aditif",
            "skala": 0.20,
            "model_tepat": "linear",
            "parameter_tepat": np.array([3.0, 5.0]),
        },
        {
            "id": "O005-LEGA-V101-CH01-P07-D2",
            "label": "f(x) = 7x",
            "x": np.linspace(0.25, 3.00, 28),
            "fungsi": lambda x: 7.0 * x,
            "bising": "aditif",
            "skala": 0.30,
            "model_tepat": "linear",
            "parameter_tepat": np.array([7.0, 0.0]),
        },
        {
            "id": "O005-LEGA-V101-CH01-P07-D3",
            "label": "f(x) = exp(3x + 7)",
            "x": np.linspace(-0.75, 0.35, 28),
            "fungsi": lambda x: np.exp(3.0 * x + 7.0),
            "bising": "multiplikatif",
            "skala": 0.015,
            "model_tepat": "eksponensial",
            "parameter_tepat": np.array([3.0, 7.0]),
        },
        {
            "id": "O005-LEGA-V101-CH01-P07-D4",
            "label": "f(x) = exp(7x)",
            "x": np.linspace(-0.30, 0.35, 28),
            "fungsi": lambda x: np.exp(7.0 * x),
            "bising": "multiplikatif",
            "skala": 0.020,
            "model_tepat": "eksponensial",
            "parameter_tepat": np.array([7.0, 0.0]),
        },
    ]

    data = []
    for spec in spesifikasi:
        x = spec["x"]
        y_tanpa_bising = spec["fungsi"](x)
        if spec["bising"] == "aditif":
            y = y_tanpa_bising + rng.normal(0.0, spec["skala"], x.size)
        else:
            y = y_tanpa_bising * np.exp(rng.normal(0.0, spec["skala"], x.size))
        data.append({**spec, "y_tanpa_bising": y_tanpa_bising, "y": y})
    return data


data_kasus = buat_data()

In [ ]:
# Pemeriksaan determinisme dan validitas data sebelum pencocokan.
data_ulang = buat_data(SEED)
assert len(data_kasus) == 4
assert all(np.array_equal(a["x"], b["x"]) for a, b in zip(data_kasus, data_ulang))
assert all(np.array_equal(a["y"], b["y"]) for a, b in zip(data_kasus, data_ulang))
assert all(item["x"].shape == (28,) and item["y"].shape == (28,) for item in data_kasus)
assert all(np.all(np.isfinite(item["y"])) for item in data_kasus)
assert all(np.all(item["y"] > 0.0) for item in data_kasus)
print("Pemeriksaan data lulus: empat kasus, masing-masing 28 titik, deterministik dan berhingga.")

## Cocokkan kedua keluarga model

`curve_fit` meminimumkan jumlah kuadrat residual. Untuk model eksponensial, garis pada $\log y$ hanya dipakai untuk menghasilkan tebakan awal; parameter akhir tetap diestimasi terhadap $y$ pada skala asal. Batas parameter yang lebar menjaga iterasi tetap numerik tanpa menentukan jawabannya terlebih dahulu.

In [ ]:
CALON_MODEL = {
    "linear": model_linear,
    "eksponensial": model_eksponensial,
}


def hitung_metrik(y, y_hat):
    residu = y - y_hat
    sse = float(np.sum(residu**2))
    sst = float(np.sum((y - np.mean(y))**2))
    return {
        "residu": residu,
        "rmse": float(np.sqrt(np.mean(residu**2))),
        "r2": 1.0 - sse / sst,
    }


def cocokkan_satu_kasus(item):
    x, y = item["x"], item["y"]
    hasil = {}

    parameter_linear, kov_linear = curve_fit(model_linear, x, y, p0=(1.0, 0.0))
    y_linear = model_linear(x, *parameter_linear)
    hasil["linear"] = {
        "parameter": parameter_linear,
        "kovarians": kov_linear,
        "y_hat": y_linear,
        **hitung_metrik(y, y_linear),
    }

    tebakan_awal = np.polyfit(x, np.log(y), 1)
    parameter_exp, kov_exp = curve_fit(
        model_eksponensial,
        x,
        y,
        p0=tebakan_awal,
        bounds=([-12.0, -12.0], [12.0, 12.0]),
        maxfev=20_000,
    )
    y_exp = model_eksponensial(x, *parameter_exp)
    hasil["eksponensial"] = {
        "parameter": parameter_exp,
        "kovarians": kov_exp,
        "y_hat": y_exp,
        **hitung_metrik(y, y_exp),
    }
    return hasil


hasil_semua = {item["id"]: cocokkan_satu_kasus(item) for item in data_kasus}

In [ ]:
print(f"{'Kasus':<25} {'Model':<14} {'a':>10} {'b':>10} {'RMSE':>12} {'R²':>10}")
print("-" * 86)
for item in data_kasus:
    for nama_model in ("linear", "eksponensial"):
        fit = hasil_semua[item["id"]][nama_model]
        a, b = fit["parameter"]
        print(
            f"{item['label']:<25} {nama_model:<14} "
            f"{a:>10.5f} {b:>10.5f} {fit['rmse']:>12.5g} {fit['r2']:>10.6f}"
        )

In [ ]:
warna = {"linear": "#0072B2", "eksponensial": "#D55E00"}
fig, axes = plt.subplots(4, 2, figsize=(13, 18), constrained_layout=True)

for baris, item in enumerate(data_kasus):
    x, y = item["x"], item["y"]
    x_rapat = np.linspace(x.min(), x.max(), 400)
    ax_kurva, ax_residu = axes[baris]

    ax_kurva.scatter(x, y, s=28, color="#222222", label="data sintetis", zorder=3)
    ax_kurva.plot(x_rapat, item["fungsi"](x_rapat), color="#009E73",
                  linewidth=2.5, linestyle="--", label="fungsi pembangkit")

    for nama_model in ("linear", "eksponensial"):
        fit = hasil_semua[item["id"]][nama_model]
        y_rapat = CALON_MODEL[nama_model](x_rapat, *fit["parameter"])
        ax_kurva.plot(x_rapat, y_rapat, color=warna[nama_model], linewidth=2,
                      label=f"fit {nama_model}")
        ax_residu.scatter(x, fit["residu"], s=25, color=warna[nama_model],
                          alpha=0.78, label=f"residual {nama_model}")

    ax_kurva.set_title(item["label"])
    ax_kurva.set_xlabel("x")
    ax_kurva.set_ylabel("y")
    ax_kurva.grid(alpha=0.25)
    ax_kurva.legend()

    ax_residu.axhline(0.0, color="#222222", linewidth=1)
    ax_residu.set_title("Perbandingan residual")
    ax_residu.set_xlabel("x")
    ax_residu.set_ylabel(r"$y-\hat{y}$")
    ax_residu.grid(alpha=0.25)
    ax_residu.legend()

fig.suptitle("Empat pencocokan kurva dan residualnya", fontsize=16)
plt.show()

## Pemeriksaan yang dapat dieksekusi

Sel berikut memeriksa bahwa optimisasi selesai dengan nilai berhingga, keluarga model yang sesuai menghasilkan RMSE terkecil, dan parameter yang dipulihkan dekat dengan parameter pembangkit. Toleransi sengaja jauh lebih besar daripada variasi numerik antarplatform, tetapi cukup ketat untuk mendeteksi perubahan algoritme atau data yang merusak hasil.

In [ ]:
toleransi_parameter = {
    "O005-LEGA-V101-CH01-P07-D1": np.array([0.20, 0.25]),
    "O005-LEGA-V101-CH01-P07-D2": np.array([0.25, 0.40]),
    "O005-LEGA-V101-CH01-P07-D3": np.array([0.15, 0.15]),
    "O005-LEGA-V101-CH01-P07-D4": np.array([0.20, 0.10]),
}

for item in data_kasus:
    hasil = hasil_semua[item["id"]]
    for fit in hasil.values():
        assert np.all(np.isfinite(fit["parameter"]))
        assert np.all(np.isfinite(fit["kovarians"]))
        assert np.all(np.isfinite(fit["residu"]))
        assert fit["rmse"] >= 0.0

    model_rmse_terkecil = min(hasil, key=lambda nama: hasil[nama]["rmse"])
    assert model_rmse_terkecil == item["model_tepat"]

    fit_tepat = hasil[item["model_tepat"]]
    selisih = np.abs(fit_tepat["parameter"] - item["parameter_tepat"])
    assert np.all(selisih < toleransi_parameter[item["id"]])
    assert fit_tepat["r2"] > 0.99

print("Semua pemeriksaan lulus: model yang tepat menang pada keempat kasus dan parameter pulih.")

## Kesimpulan

Pencocokan memuaskan bukan sekadar kurva yang tampak melewati banyak titik. Pada dua data linear, model linear memperoleh RMSE lebih kecil dan residualnya tersebar rapat di sekitar nol; model eksponensial meninggalkan pola melengkung. Pada dua data eksponensial terjadi kebalikannya. Estimasi $a$ dan $b$ dari keluarga yang sesuai juga dekat dengan nilai pembangkit $(3,5)$, $(7,0)$, $(3,7)$, dan $(7,0)$. Jadi, perangkat optimisasi tidak dapat menggantikan pemilihan bentuk model: grafik residual dan pengetahuan tentang mekanisme tetap perlu diperiksa.

**Deskripsi panjang gambar:** Gambar terdiri atas empat baris dan dua kolom. Setiap baris mewakili satu fungsi. Panel kiri menampilkan titik data hitam, fungsi pembangkit hijau putus-putus, fit linear biru, dan fit eksponensial jingga. Untuk dua fungsi linear, garis biru hampir bertumpuk dengan fungsi pembangkit sedangkan kurva jingga menyimpang secara sistematis. Untuk dua fungsi eksponensial, kurva jingga hampir bertumpuk dengan fungsi pembangkit sedangkan garis biru menyimpang terutama di ujung rentang. Panel kanan menampilkan residual kedua model terhadap $x$ dan garis nol. Residual model yang bentuknya sesuai membentuk pita sempit di sekitar nol; residual model yang tidak sesuai memperlihatkan pola terstruktur.